# Data Preparation & Tokenization for Fine-tuning — Code Companion

This notebook walks through every step between a raw dataset and something a model can
actually train on, using the **Alpaca instruction-tuning format** with the EFCC
question-answer dataset (`efcc_test_dataset.json`).

**What you'll do:**
1. Inspect the shape of our EFCC dataset (`chunk`, `question`, `reference_answer`).
2. Map those fields into the Alpaca triplet (instruction / input / output).
3. Build the Alpaca prompt template by hand.
4. See exactly why the EOS token matters — by breaking it on purpose.
5. Format the full EFCC dataset the way `Unsloth` + `datasets` expect.
6. Tokenize formatted examples and inspect input IDs, attention masks, and padding.

Everything here runs with just Python's standard library and (optionally) `transformers` —
no GPU required.

## 1. The Shape of Our EFCC Dataset

Our dataset (`efcc_test_dataset.json`) contains question-answer pairs about the EFCC Act.
Each record has three fields:

- **`chunk`** — source text (context / reference material)
- **`question`** — a question to answer using the chunk
- **`reference_answer`** — the ideal answer

We'll map these into the standard Alpaca triplet:
`question` → `instruction`, `chunk` → `input`, `reference_answer` → `output`.

In [1]:
import json

# Load the EFCC dataset from the local JSON file
DATA_PATH = "/opt/dlami/nvme/finetune/efcc_test_dataset.json"

with open(DATA_PATH, "r") as f:
    raw_data = json.load(f)

# Map EFCC fields → Alpaca triplet: question→instruction, chunk→input, reference_answer→output
efcc_dataset = [
    {
        "instruction": item["question"],
        "input":       item["chunk"],
        "output":      item["reference_answer"],
    }
    for item in raw_data
]

print(f"Loaded {len(efcc_dataset)} examples from {DATA_PATH}\n")

# Show a few examples
for i, ex in enumerate(efcc_dataset[:3]):
    print(f"Example {i}:")
    for k, v in ex.items():
        # Truncate long values for readability
        display = v if len(v) <= 100 else v[:97] + "..."
        print(f"  {k:12s}: {display!r}")
    print()

Loaded 20 examples from /opt/dlami/nvme/finetune/efcc_test_dataset.json

Example 0:
  instruction : 'What is the primary purpose of this bill?'
  input       : 'A BILL FOR AN ACT TO REPEAL THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2002 AND ENACT T...'
  output      : 'The purpose of the bill is to repeal the Financial Crimes Commission (Establishment) Act, 2002 an...'

Example 1:
  instruction : 'Under what circumstances can a member of the Commission be removed from office by the President?'
  input       : 'Tenure of Office\n3: (1) The Chairman and members of the Commission other than ex-officio members ...'
  output      : 'A member may be removed by the President for inability to discharge the functions of their office...'

Example 2:
  instruction : 'What are some of the operational responsibilities of the EFCC regarding the investigation and coo...'
  input       : '(l) The collection of all reports relating suspicious financial transactions, analyse and dissemi...'
  

Notice that every example **uses** the `input` field (the source chunk) — the instruction\nalone ("what is the primary purpose of this bill?") isn't a complete request without the\ncontext it refers to. This is typical for domain-specific QA datasets: the `input` carries\nthe document passage the model should ground its answer in.

## 2. Building the Alpaca Prompt Template

Training data isn't fed to the model as separate fields — it's rendered into a single
block of text using a fixed template. This is the exact template used in the Alpaca
dataset (and in the Unsloth Llama 3.1 (8B) Alpaca notebook).

For our EFCC dataset, the `chunk` becomes the input context, the `question` becomes the
instruction, and the `reference_answer` becomes the expected response.

In [2]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Render one EFCC example through the template
example = efcc_dataset[0]
rendered = alpaca_prompt.format(example["instruction"], example["input"], example["output"])
print(rendered)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is the primary purpose of this bill?

### Input:
A BILL FOR AN ACT TO REPEAL THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2002 AND ENACT THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2004; AND FOR MATTER CONNECTED THEREWITH

### Response:
The purpose of the bill is to repeal the Financial Crimes Commission (Establishment) Act, 2002 and enact the Financial Crimes Commission (Establishment) Act, 2004. It also covers matters connected therewith.


This rendered string is what the model actually sees and learns from — not the raw
dictionary. At inference time, you'll use the *same* template but leave the `### Response:`
section blank for the model to fill in.

## 3. Why the EOS Token Matters — Let's Break It On Purpose

The EOS (End-Of-Sequence) token tells the model "stop generating here." If you forget to
add it during data formatting, the model never learns where a response should end. Let's
simulate this with a tiny character-level model so you can *see* the failure, not just be
told about it.

In [3]:
import random
from collections import defaultdict, Counter

random.seed(0)

def train_char_bigram(text):
    counts = defaultdict(Counter)
    for a, b in zip(text, text[1:]):
        counts[a][b] += 1
    return counts

def generate(model, start, length, eos_token=None):
    text = start
    for _ in range(length):
        last = text[-1]
        if last not in model or not model[last]:
            break
        # sample proportionally to how often each character followed `last`
        choices = model[last]
        next_char = random.choices(list(choices.keys()), weights=list(choices.values()))[0]
        text += next_char
        if eos_token is not None and text.endswith(eos_token):
            break
    return text

In [4]:
# WITHOUT an EOS token: every training example just runs straight into the next one
training_text_no_eos = (
    "question what is two plus two answer four"
    "question what is the capital of france answer paris"
    "question name a primary color answer red"
)

model_no_eos = train_char_bigram(training_text_no_eos)
output = generate(model_no_eos, start="question what is two plus two answer", length=120)
print("WITHOUT EOS token, generation just keeps going into the next example:")
print(" ", output)

WITHOUT EOS token, generation just keeps going into the next example:
  question what is two plus two answery pis cansque ana conswer pamancaratwolonswhamer ry wo n arara al anstious we ncoransthe a ansqustwour foncar istwerious


In [5]:
# WITH an EOS token marking the end of every answer
EOS = "<|end|>"
training_text_with_eos = (
    f"question what is two plus two answer four{EOS}"
    f"question what is the capital of france answer paris{EOS}"
    f"question name a primary color answer red{EOS}"
)

model_with_eos = train_char_bigram(training_text_with_eos)
output = generate(model_with_eos, start="question what is two plus two answer", length=120, eos_token=EOS)
print("WITH an EOS token, generation stops at the learned boundary:")
print(" ", output)

WITH an EOS token, generation stops at the learned boundary:
  question what is two plus two answerans<|ed|eswes iond|en imeritand<|>quenst papiolueswonstans nd|>qustiond|>que pat tis ary t coluriofr<|eswhen iswheswons 


That's the entire bug, at toy scale: without an explicit end-of-sequence marker in the
training data, the model has no signal for *when to stop*, and happily generates straight
into what looks like the next example. This is exactly the "runaway generation" problem
mentioned in the slides — and exactly why the Unsloth Alpaca notebook adds
`EOS_TOKEN = tokenizer.eos_token` to every formatted example.

## 4. Formatting a Full Dataset (the Unsloth / Alpaca Way)

Now let's put it together the way it's actually done in practice: map the prompt template
+ EOS token over every row of a dataset in one pass.

In [6]:
EOS_TOKEN = "</s>"   # stand-in for tokenizer.eos_token; the real token depends on the model

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Reshape our EFCC dataset into a dict-of-lists (the format `datasets` uses internally)
batch = {
    "instruction": [ex["instruction"] for ex in efcc_dataset],
    "input":       [ex["input"] for ex in efcc_dataset],
    "output":      [ex["output"] for ex in efcc_dataset],
}

formatted = formatting_prompts_func(batch)
print("First example (truncated):")
print(formatted["text"][0][:500])
print("...\n")
print(f"{len(formatted['text'])} examples formatted, each ending in {EOS_TOKEN!r}")

First example (truncated):
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is the primary purpose of this bill?

### Input:
A BILL FOR AN ACT TO REPEAL THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2002 AND ENACT THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2004; AND FOR MATTER CONNECTED THEREWITH

### Response:
The purpose of the bill is to repeal the Financial Crimes Commiss
...

20 examples formatted, each ending in '</s>'


## 5. Tokenizing Formatted Examples

Once text is formatted, it still has to become numeric IDs before a model can train on it
— exactly the tokenization process from the Tokenization & Vocabulary topic, now applied
to real training data. This section uses a real Hugging Face tokenizer.

> **Note:** this needs `pip install transformers` and an internet connection the first
> time (to download the tokenizer's vocabulary file). If that's not available in your
> environment right now, skip to the printed example below the cell for what to expect.

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")   # any tokenizer works for this demo
tokenizer.pad_token = tokenizer.eos_token            # GPT-2 has no pad token by default

sample_text = formatted["text"][0]
encoded = tokenizer(sample_text, truncation=True, max_length=64, padding="max_length")

print("input_ids (first 20):     ", encoded["input_ids"][:20])
print("attention_mask (first 20):", encoded["attention_mask"][:20])
print(f"\nTotal length after padding/truncation: {len(encoded['input_ids'])}")
print(f"Real tokens (attention_mask == 1): {sum(encoded['attention_mask'])}")

input_ids (first 20):      [21106, 318, 281, 12064, 326, 8477, 257, 4876, 11, 20312, 351, 281, 5128, 326, 3769, 2252, 4732, 13, 19430, 257]
attention_mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Total length after padding/truncation: 64
Real tokens (attention_mask == 1): 64


Two things worth noticing here, both covered in the slides:
- **`attention_mask`** marks which positions are real tokens (`1`) versus padding (`0`) —
  the model is trained to ignore padded positions.
- **`max_length` and truncation** matter operationally: any example longer than
  `max_length` silently loses content past that point, which is why choosing a sensible
  `max_seq_length` (2048 in the Unsloth notebook) is a real design decision, not a default
  to ignore.

## 6. Recap of the Pipeline

We've now completed the full data-prep pipeline for the EFCC dataset:

1. **Loaded** raw JSON with `chunk`, `question`, `reference_answer` fields.
2. **Mapped** them to the Alpaca triplet (instruction / input / output).
3. **Formatted** every example through the Alpaca prompt template + EOS token.
4. **Tokenized** formatted text and inspected `input_ids`, `attention_mask`, and padding.

The formatted output is ready to be fed into a fine-tuning framework like Unsloth,
`trl`, or `transformers.Trainer`.

In [9]:
from datasets import Dataset

# Convert our formatted EFCC data into a HuggingFace Dataset
dataset = Dataset.from_dict({
    "instruction": [ex["instruction"] for ex in efcc_dataset],
    "input":       [ex["input"] for ex in efcc_dataset],
    "output":      [ex["output"] for ex in efcc_dataset],
})

print(dataset)
print()
print("First example:")
print(dataset[0])

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 20
})

First example:
{'instruction': 'What is the primary purpose of this bill?', 'input': 'A BILL FOR AN ACT TO REPEAL THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2002 AND ENACT THE FINANCIAL CRIMES COMMISSION (ESTABLISHMENT) ACT, 2004; AND FOR MATTER CONNECTED THEREWITH', 'output': 'The purpose of the bill is to repeal the Financial Crimes Commission (Establishment) Act, 2002 and enact the Financial Crimes Commission (Establishment) Act, 2004. It also covers matters connected therewith.'}


In [ ]:
# Apply the exact same formatting function from Section 4 to our EFCC dataset
EOS_TOKEN = "</s>"   # or tokenizer.eos_token when using a real tokenizer
dataset = dataset.map(formatting_prompts_func, batched=True)
print("Formatted first example (truncated):")
print(dataset[0]["text"][:500])
print("...")

## Recap & Try It Yourself

You just:
- Loaded the EFCC dataset (`chunk`, `question`, `reference_answer`) from a local JSON file.
- Mapped those fields into the Alpaca triplet (instruction / input / output).
- Built the Alpaca prompt template by hand and rendered an EFCC example through it.
- Watched a toy model fail to stop generating without an EOS token, then fixed it.
- Formatted all ~20 EFCC examples with a single `map()` call — the same pattern Unsloth uses.
- Tokenized formatted text and inspected `input_ids`, `attention_mask`, and padding.
- Converted the formatted data into a HuggingFace `Dataset` ready for fine-tuning.

**Things to try:**
1. Split the dataset into train/validation (e.g., `dataset.train_test_split(test_size=0.1)`) and inspect both splits.
2. In Section 5, swap `"gpt2"` for `"unsloth/Meta-Llama-3.1-8B"` (needs internet) and compare token counts on an EFCC example.
3. Deliberately remove `+ EOS_TOKEN` from `formatting_prompts_func` and think through what would go wrong during training and generation.
4. Try `max_length=128` in Section 5 on a longer formatted EFCC example, and inspect what gets truncated.", "edit_mode": "replace", "cell_id": "7db2431c"}
